# 1 · 15-minute targets
Our proposal predicts **15 minutes** ahead; the professor's pipeline uses 30. The predictors don't depend on the horizon, so we reuse his `gold_station_predictors` table and only rebuild the targets, with his `build_targets()` function and `horizon_minutes=15`.

**Target** (`bike_target`, `dock_target`): 1 if a station that has bikes (docks) now runs out at any poll in the next 15 minutes, 0 if it doesn't, and null if it's already out, not operating, or the next 15 minutes aren't fully observed.

**Needs:** `free_edition_models/3-gold-features-FE` run in production first. **Writes:** `team1_gold_targets_15`.

In [ ]:
# Settings and shared code. mvp_config.py and mvp_lib.py sit next to this notebook;
# station_features.py is the professor's file in free_edition_models/.
import mvp_config as cfg
import mvp_lib as lib

lib.use_station_features()
from pyspark.sql import functions as F
from station_features import build_targets

spark.conf.set("spark.sql.session.timeZone", "UTC")

# Interactive runs default to dev (calculate and show, write nothing). Jobs pass production.
dbutils.widgets.dropdown("run_mode", "dev", ["dev", "production"])
PUBLISH = dbutils.widgets.get("run_mode") == "production"
print(f"run_mode={dbutils.widgets.get('run_mode')} · reads {cfg.PREDICTORS} · writes {cfg.TARGETS if PUBLISH else '(nothing)'}")

In [ ]:
# Build the 15-minute targets from the existing predictors table.
predictors = spark.table(cfg.PREDICTORS)
targets = build_targets(predictors, horizon_minutes=cfg.HORIZON_MIN,
                        max_gap_minutes=cfg.MAX_OBSERVATION_GAP_MIN)

if PUBLISH:
    (targets.write.format("delta").mode("overwrite").option("overwriteSchema", "true")
            .saveAsTable(cfg.TARGETS))
    targets = spark.table(cfg.TARGETS)
else:
    print("dev: table not written")

In [ ]:
# How rare are new stockouts within 15 minutes? Compare with the 30-minute table if it exists.
summary = targets.agg(
    F.lit(cfg.HORIZON_MIN).alias("horizon_min"),
    F.count("*").alias("polls"),
    F.count("bike_target").alias("bike_labeled"),
    F.avg("bike_target").alias("bike_onset_share"),
    F.count("dock_target").alias("dock_labeled"),
    F.avg("dock_target").alias("dock_onset_share"),
)
thirty = f"{cfg.SCHEMA}.gold_station_targets"
if spark.catalog.tableExists(thirty):
    summary = summary.unionByName(spark.table(thirty).agg(
        F.lit(30).alias("horizon_min"),
        F.count("*").alias("polls"),
        F.count("bike_target").alias("bike_labeled"),
        F.avg("bike_target").alias("bike_onset_share"),
        F.count("dock_target").alias("dock_labeled"),
        F.avg("dock_target").alias("dock_onset_share"),
    ))
display(summary)